# Advanced Optimizer Comparison: From AdamW to Modern Alternatives

**Historical Context**: While Adam (2014) and AdamW (2017) have been the workhorses of deep learning, recent years have seen exciting developments in optimizer design. This notebook explores modern optimizers that offer better performance, memory efficiency, or speed.

**Key Innovations**:
- **AdamW** (2017): Decoupled weight decay for better generalization
- **8-bit Optimizers** (2021): Quantized optimizer states for memory efficiency
- **Lion** (2023): Simpler, more memory-efficient alternative to Adam
- **Sophia** (2023): Second-order optimizer using Hessian information

**Learning Objectives**:
- Understand AdamW and weight decay
- Learn Lion optimizer (2023)
- Explore Sophia optimizer (2023)
- Master 8-bit optimizers for memory savings
- Compare memory-efficient optimizers
- Benchmark performance across optimizers
- Know when to use which optimizer

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time
from typing import Dict, List, Tuple, Optional
import pandas as pd

# Check for bitsandbytes (8-bit optimizers)
try:
    import bitsandbytes as bnb
    BNB_AVAILABLE = True
    print(f"bitsandbytes version: {bnb.__version__}")
except ImportError:
    BNB_AVAILABLE = False
    print("bitsandbytes not installed. Install with: pip install bitsandbytes")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nPyTorch version: {torch.__version__}")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 1. AdamW: The Modern Standard (2017)

**Original Adam** (Kingma & Ba, 2014):
```
m_t = β₁ * m_{t-1} + (1-β₁) * g_t          # 1st moment (momentum)
v_t = β₂ * v_{t-1} + (1-β₂) * g_t²         # 2nd moment (variance)
θ_t = θ_{t-1} - α * m_t / (√v_t + ε)       # Update
```

**AdamW** (Loshchilov & Hutter, 2017):
- **Key Innovation**: Decoupled weight decay
- Original Adam: weight decay applied to gradients (incorrect!)
- AdamW: weight decay applied directly to parameters
```
θ_t = θ_{t-1} - α * m_t / (√v_t + ε) - α * λ * θ_{t-1}  # Separate weight decay
```

**Why AdamW is Better**:
- Better generalization
- More stable training
- Standard for transformers

**Memory per Parameter**:
- Parameters: 4 bytes (FP32)
- Momentum (m): 4 bytes
- Variance (v): 4 bytes
- **Total: 12 bytes per parameter**

In [ ]:
def visualize_optimizer_memory():
    """
    Visualize memory usage of different optimizers.
    """
    optimizers = {
        'SGD': {'params': 4, 'momentum': 0, 'variance': 0},
        'SGD\n+Momentum': {'params': 4, 'momentum': 4, 'variance': 0},
        'Adam/AdamW': {'params': 4, 'momentum': 4, 'variance': 4},
        '8-bit Adam': {'params': 4, 'momentum': 1, 'variance': 1},
        'Lion': {'params': 4, 'momentum': 4, 'variance': 0},
        '8-bit Lion': {'params': 4, 'momentum': 1, 'variance': 0},
    }
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    names = list(optimizers.keys())
    x = np.arange(len(names))
    width = 0.6
    
    # Stack components
    params = [optimizers[name]['params'] for name in names]
    momentum = [optimizers[name]['momentum'] for name in names]
    variance = [optimizers[name]['variance'] for name in names]
    
    p1 = ax.bar(x, params, width, label='Parameters', alpha=0.8, color='#FF6B6B')
    p2 = ax.bar(x, momentum, width, bottom=params, label='Momentum', alpha=0.8, color='#4ECDC4')
    p3 = ax.bar(x, variance, width, bottom=np.array(params)+np.array(momentum), 
               label='Variance', alpha=0.8, color='#45B7D1')
    
    # Add total labels
    for i, name in enumerate(names):
        total = params[i] + momentum[i] + variance[i]
        ax.text(i, total + 0.3, f'{total} bytes', ha='center', va='bottom', 
               fontsize=10, fontweight='bold')
    
    ax.set_ylabel('Memory per Parameter (bytes)', fontsize=12)
    ax.set_title('Optimizer Memory Usage Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.legend(fontsize=11, loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 15)
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/optimizer_memory.png',
                dpi=150, bbox_inches='tight')
    plt.show()

visualize_optimizer_memory()

# Calculate memory for different model sizes
def calculate_optimizer_memory(num_params: int, optimizer_type: str) -> float:
    """Calculate optimizer memory in GB."""
    bytes_per_param = {
        'SGD': 4,
        'SGD+Momentum': 8,
        'Adam': 12,
        'AdamW': 12,
        '8-bit Adam': 6,
        'Lion': 8,
        '8-bit Lion': 5,
    }
    return (num_params * bytes_per_param[optimizer_type]) / (1024**3)

print("\nOptimizer Memory Requirements:")
print("="*100)
print(f"{'Model Size':<20} {'Adam/AdamW':<15} {'8-bit Adam':<15} {'Lion':<15} {'8-bit Lion':<15}")
print("="*100)

model_sizes = [
    ('GPT-2 (124M)', 124_000_000),
    ('GPT-2 Large (774M)', 774_000_000),
    ('LLaMA-7B', 7_000_000_000),
    ('LLaMA-13B', 13_000_000_000),
    ('LLaMA-70B', 70_000_000_000),
]

for name, num_params in model_sizes:
    adam_mem = calculate_optimizer_memory(num_params, 'Adam')
    adam8_mem = calculate_optimizer_memory(num_params, '8-bit Adam')
    lion_mem = calculate_optimizer_memory(num_params, 'Lion')
    lion8_mem = calculate_optimizer_memory(num_params, '8-bit Lion')
    
    print(f"{name:<20} {adam_mem:<15.2f} {adam8_mem:<15.2f} {lion_mem:<15.2f} {lion8_mem:<15.2f}")

print("\nNote: Values in GB. Includes momentum and variance states.")

## 2. Lion Optimizer (2023)

**Paper**: "Symbolic Discovery of Optimization Algorithms" (Chen et al., Google, 2023)

**Key Innovation**: Discovered by program search, Lion is simpler than Adam:
- Only uses **momentum** (no variance)
- Sign-based updates (like signSGD)
- 2x less memory than Adam

**Algorithm**:
```
c_t = β₁ * m_{t-1} + (1-β₁) * g_t          # Interpolate
θ_t = θ_{t-1} - α * sign(c_t)              # Sign update
m_t = β₂ * m_{t-1} + (1-β₂) * g_t          # Update momentum
```

**Advantages**:
- 50% less memory than Adam (8 bytes vs 12 bytes per param)
- Often better performance than Adam
- More stable, less sensitive to hyperparameters
- Faster convergence in some cases

**Typical Hyperparameters**:
- Learning rate: 10x smaller than Adam (e.g., 3e-5 instead of 3e-4)
- β₁ = 0.9, β₂ = 0.99 (default)
- Weight decay: 10x larger than Adam

In [ ]:
# Lion optimizer implementation
class Lion(torch.optim.Optimizer):
    """
    Lion optimizer.
    Paper: https://arxiv.org/abs/2302.06675
    """
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        if lr <= 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta parameter: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta parameter: {betas[1]}")
        if weight_decay < 0.0:
            raise ValueError(f"Invalid weight_decay value: {weight_decay}")
        
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)
    
    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                
                grad = p.grad
                if grad.is_sparse:
                    raise RuntimeError('Lion does not support sparse gradients')
                
                state = self.state[p]
                
                # State initialization
                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)
                
                exp_avg = state['exp_avg']
                beta1, beta2 = group['betas']
                
                # Weight decay
                if group['weight_decay'] > 0.0:
                    p.mul_(1 - group['lr'] * group['weight_decay'])
                
                # Update with interpolation
                update = exp_avg.mul(beta1).add(grad, alpha=1-beta1)
                
                # Apply sign update
                p.add_(torch.sign(update), alpha=-group['lr'])
                
                # Update momentum
                exp_avg.mul_(beta2).add_(grad, alpha=1-beta2)
        
        return loss

# Test Lion optimizer
print("Lion Optimizer Implementation:")
print("="*70)
print("Key Properties:")
print("  - Uses only momentum (no variance)")
print("  - Sign-based updates")
  - 2x less memory than Adam")
print("  - Often better performance")
print("\nRecommended Settings:")
print("  - Learning rate: 10x smaller than Adam")
print("  - Weight decay: 10x larger than Adam")
print("  - Betas: (0.9, 0.99) - default")

# Example usage
model = nn.Linear(100, 10).to(device)
optimizer = Lion(model.parameters(), lr=1e-4, weight_decay=0.01)

# Compare with AdamW
print("\nExample:")
print("  Lion:  lr=1e-4, weight_decay=0.1")
print("  AdamW: lr=1e-3, weight_decay=0.01")

## 3. Sophia Optimizer (2023)

**Paper**: "Sophia: A Scalable Stochastic Second-order Optimizer" (Liu et al., Stanford, 2023)

**Key Innovation**: Uses **second-order information** (Hessian diagonal) efficiently:
- Estimates Hessian diagonal cheaply
- Adapts learning rate per-parameter based on curvature
- 2x faster convergence than Adam on language models

**Algorithm**:
```
m_t = β₁ * m_{t-1} + (1-β₁) * g_t              # Momentum
h_t = EMA(diag(Hessian))                        # Hessian diagonal estimate
θ_t = θ_{t-1} - α * m_t / max(h_t, ε)           # Curvature-aware update
```

**Advantages**:
- 2x faster convergence (fewer steps to same loss)
- Better for large language models
- Adaptive to per-parameter geometry

**Disadvantages**:
- Requires Hessian computation (extra forward-backward pass)
- ~50% more compute per step
- But 2x fewer steps → net speedup!

**When to Use**:
- Training large language models
- Long training runs where convergence speed matters
- When compute budget is flexible

In [ ]:
# Simplified Sophia implementation (Sophia-H variant)
class SophiaG(torch.optim.Optimizer):
    """
    Sophia-G optimizer (uses Gauss-Newton-Bartlett estimate).
    Simplified implementation for demonstration.
    Paper: https://arxiv.org/abs/2305.14342
    """
    def __init__(self, params, lr=1e-4, betas=(0.965, 0.99), rho=0.04, weight_decay=1e-4, eps=1e-12):
        defaults = dict(lr=lr, betas=betas, rho=rho, weight_decay=weight_decay, eps=eps)
        super().__init__(params, defaults)
    
    @torch.no_grad()
    def step(self, closure=None, bs=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                
                grad = p.grad
                state = self.state[p]
                
                # State initialization
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['hessian'] = torch.zeros_like(p)
                
                exp_avg, hessian = state['exp_avg'], state['hessian']
                beta1, beta2 = group['betas']
                state['step'] += 1
                
                # Weight decay
                if group['weight_decay'] > 0:
                    p.mul_(1 - group['lr'] * group['weight_decay'])
                
                # Update momentum
                exp_avg.mul_(beta1).add_(grad, alpha=1-beta1)
                
                # Update Hessian estimate (simplified)
                # In practice, this would use a separate forward-backward pass
                hessian.mul_(beta2).add_(grad.pow(2), alpha=1-beta2)
                
                # Bias correction
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                
                # Corrected estimates
                exp_avg_corr = exp_avg / bias_correction1
                hessian_corr = hessian / bias_correction2
                
                # Clipped update
                update = exp_avg_corr / (hessian_corr.clamp(min=group['eps']) + group['rho'])
                update = torch.clamp(update, -1, 1)  # Clip for stability
                
                p.add_(update, alpha=-group['lr'])
        
        return loss

print("Sophia Optimizer:")
print("="*70)
print("Key Features:")
print("  - Uses second-order information (Hessian diagonal)")
print("  - 2x faster convergence on LLMs")
print("  - ~50% more compute per step")
print("  - Net speedup due to fewer steps")
print("\nBest For:")
print("  - Large language model pre-training")
print("  - Long training runs")
print("  - When convergence speed > compute cost")
print("\nTypical Settings:")
print("  - lr=1e-4, betas=(0.965, 0.99)")
print("  - rho=0.04, weight_decay=1e-4")

## 4. 8-bit Optimizers (2021)

**Paper**: "8-bit Optimizers via Block-wise Quantization" (Dettmers et al., 2021)

**Key Innovation**: Quantize optimizer states (momentum, variance) to 8-bit:
- Reduces memory by 75% for optimizer states
- Uses block-wise quantization to maintain precision
- Negligible performance loss

**Memory Savings**:
```
Adam (32-bit):
  - Momentum: 4 bytes per param
  - Variance: 4 bytes per param
  - Total: 8 bytes per param

8-bit Adam:
  - Momentum: 1 byte per param
  - Variance: 1 byte per param
  - Total: 2 bytes per param
  - 75% memory reduction!
```

**Implementation**: Use `bitsandbytes` library

In [ ]:
# Example: 8-bit optimizer usage
if BNB_AVAILABLE:
    print("8-bit Optimizer Examples:")
    print("="*70)
    
    # Create a model
    model = nn.Sequential(
        nn.Linear(512, 2048),
        nn.ReLU(),
        nn.Linear(2048, 512),
    ).to(device)
    
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}")
    
    # 1. Standard AdamW (32-bit)
    optimizer_32bit = torch.optim.AdamW(model.parameters(), lr=1e-4)
    
    # 2. 8-bit AdamW
    optimizer_8bit = bnb.optim.AdamW8bit(model.parameters(), lr=1e-4)
    
    # 3. Other 8-bit optimizers
    # optimizer_8bit = bnb.optim.Adam8bit(model.parameters(), lr=1e-4)
    # optimizer_8bit = bnb.optim.Lion8bit(model.parameters(), lr=1e-5)
    
    print("\nOptimizer Memory Comparison:")
    print("-"*70)
    
    adam32_memory = (num_params * 8) / (1024**2)  # 8 bytes per param
    adam8_memory = (num_params * 2) / (1024**2)   # 2 bytes per param
    
    print(f"AdamW 32-bit states: {adam32_memory:.2f} MB")
    print(f"AdamW 8-bit states:  {adam8_memory:.2f} MB")
    print(f"Savings: {(1 - adam8_memory/adam32_memory)*100:.1f}%")
    
    # Usage is identical
    x = torch.randn(8, 512, device=device)
    
    output = model(x)
    loss = output.mean()
    loss.backward()
    optimizer_8bit.step()
    optimizer_8bit.zero_grad()
    
    print("\nUsage: Identical to standard PyTorch optimizers!")
    
else:
    print("Install bitsandbytes to use 8-bit optimizers:")
    print("  pip install bitsandbytes")
    print("\nSupported optimizers:")
    print("  - bnb.optim.Adam8bit")
    print("  - bnb.optim.AdamW8bit")
    print("  - bnb.optim.Lion8bit")
    print("  - bnb.optim.LAMB8bit")
    print("  - And more...")

# Memory savings table
print("\n\n8-bit Optimizer Memory Savings:")
print("="*80)

model_sizes = [
    ('GPT-2 (124M)', 124_000_000),
    ('LLaMA-7B', 7_000_000_000),
    ('LLaMA-13B', 13_000_000_000),
    ('LLaMA-70B', 70_000_000_000),
]

data = []
for name, num_params in model_sizes:
    adam32 = (num_params * 8) / (1024**3)
    adam8 = (num_params * 2) / (1024**3)
    savings_gb = adam32 - adam8
    savings_pct = (1 - adam8/adam32) * 100
    
    data.append({
        'Model': name,
        '32-bit (GB)': f'{adam32:.2f}',
        '8-bit (GB)': f'{adam8:.2f}',
        'Savings (GB)': f'{savings_gb:.2f}',
        'Savings (%)': f'{savings_pct:.1f}%',
    })

df = pd.DataFrame(data)
print(df.to_string(index=False))
print("="*80)

## 5. Optimizer Benchmark Comparison

Let's compare different optimizers on a real training task:

In [ ]:
# Simple benchmark model
class BenchmarkModel(nn.Module):
    def __init__(self, hidden_dim=512, num_layers=4):
        super().__init__()
        layers = []
        for _ in range(num_layers):
            layers.extend([
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),
            ])
        self.layers = nn.Sequential(*layers)
        self.output = nn.Linear(hidden_dim, 10)  # 10 classes
    
    def forward(self, x):
        x = self.layers(x)
        return self.output(x)

def benchmark_optimizer(optimizer_class, optimizer_kwargs, num_steps=100):
    """Benchmark an optimizer."""
    model = BenchmarkModel().to(device)
    optimizer = optimizer_class(model.parameters(), **optimizer_kwargs)
    
    # Dummy data
    batch_size = 32
    
    losses = []
    
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    
    start_time = time.time()
    
    for step in range(num_steps):
        # Create random data
        x = torch.randn(batch_size, 512, device=device)
        y = torch.randint(0, 10, (batch_size,), device=device)
        
        # Forward
        output = model(x)
        loss = F.cross_entropy(output, y)
        
        # Backward
        loss.backward()
        
        # Optimizer step
        optimizer.step()
        optimizer.zero_grad()
        
        losses.append(loss.item())
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    elapsed = time.time() - start_time
    
    if device.type == 'cuda':
        peak_memory = torch.cuda.max_memory_allocated() / (1024**2)
    else:
        peak_memory = 0
    
    return {
        'final_loss': losses[-1],
        'avg_loss_last_10': np.mean(losses[-10:]),
        'time_per_step': elapsed / num_steps * 1000,  # ms
        'peak_memory': peak_memory,
        'losses': losses,
    }

# Run benchmarks
print("Running optimizer benchmarks...")
print("="*70)

optimizers_to_test = [
    ('SGD+Momentum', torch.optim.SGD, {'lr': 1e-2, 'momentum': 0.9}),
    ('Adam', torch.optim.Adam, {'lr': 1e-3}),
    ('AdamW', torch.optim.AdamW, {'lr': 1e-3, 'weight_decay': 0.01}),
    ('Lion', Lion, {'lr': 1e-4, 'weight_decay': 0.1}),
]

if BNB_AVAILABLE and device.type == 'cuda':
    optimizers_to_test.extend([
        ('AdamW 8-bit', bnb.optim.AdamW8bit, {'lr': 1e-3, 'weight_decay': 0.01}),
        ('Lion 8-bit', bnb.optim.Lion8bit, {'lr': 1e-4, 'weight_decay': 0.1}),
    ])

results = {}
for name, opt_class, opt_kwargs in optimizers_to_test:
    print(f"Testing {name}...")
    results[name] = benchmark_optimizer(opt_class, opt_kwargs, num_steps=100)

print("\nBenchmark complete!")

In [ ]:
# Visualize results
def plot_benchmark_results(results):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Training curves
    ax = axes[0, 0]
    for name, result in results.items():
        ax.plot(result['losses'], label=name, linewidth=2, alpha=0.8)
    ax.set_xlabel('Step', fontsize=11)
    ax.set_ylabel('Loss', fontsize=11)
    ax.set_title('Training Loss', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    
    # Final loss comparison
    ax = axes[0, 1]
    names = list(results.keys())
    final_losses = [results[name]['avg_loss_last_10'] for name in names]
    bars = ax.barh(names, final_losses, alpha=0.8, color='#4ECDC4')
    ax.set_xlabel('Average Loss (last 10 steps)', fontsize=11)
    ax.set_title('Final Performance', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, (bar, loss) in enumerate(zip(bars, final_losses)):
        ax.text(loss, i, f' {loss:.4f}', va='center', fontsize=9)
    
    # Speed comparison
    ax = axes[1, 0]
    times = [results[name]['time_per_step'] for name in names]
    bars = ax.barh(names, times, alpha=0.8, color='#FF6B6B')
    ax.set_xlabel('Time per Step (ms)', fontsize=11)
    ax.set_title('Speed Comparison', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    for i, (bar, t) in enumerate(zip(bars, times)):
        ax.text(t, i, f' {t:.2f}ms', va='center', fontsize=9)
    
    # Memory comparison
    ax = axes[1, 1]
    if device.type == 'cuda':
        memories = [results[name]['peak_memory'] for name in names]
        bars = ax.barh(names, memories, alpha=0.8, color='#45B7D1')
        ax.set_xlabel('Peak Memory (MB)', fontsize=11)
        ax.set_title('Memory Usage', fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        
        for i, (bar, mem) in enumerate(zip(bars, memories)):
            ax.text(mem, i, f' {mem:.0f}MB', va='center', fontsize=9)
    else:
        ax.text(0.5, 0.5, 'GPU required\nfor memory tracking', 
               ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/optimizer_benchmark.png',
                dpi=150, bbox_inches='tight')
    plt.show()

plot_benchmark_results(results)

# Summary table
summary_data = []
for name, result in results.items():
    summary_data.append({
        'Optimizer': name,
        'Final Loss': f"{result['avg_loss_last_10']:.4f}",
        'Time/Step (ms)': f"{result['time_per_step']:.2f}",
        'Memory (MB)': f"{result['peak_memory']:.0f}" if device.type == 'cuda' else 'N/A',
    })

df = pd.DataFrame(summary_data)
print("\nBenchmark Summary:")
print("="*80)
print(df.to_string(index=False))
print("="*80)

## 6. When to Use Which Optimizer

### Decision Guide:

```
Training transformers / LLMs?
├─ Yes → Go to (A)
└─ No → Go to (B)

(A) Transformer/LLM Training:
    ├─ Memory constrained?
    │  ├─ Yes → 8-bit AdamW or Lion 8-bit
    │  └─ No → Go to (A1)
    ├─ (A1) Long training run (>1 week)?
    │  ├─ Yes → Sophia (2x faster convergence)
    │  └─ No → Go to (A2)
    └─ (A2) Standard case
       ├─ Want less memory? → Lion
       └─ Default → AdamW

(B) Other Tasks:
    ├─ CNNs / Computer Vision?
    │  └─ AdamW or SGD+Momentum
    ├─ Small models?
    │  └─ AdamW (fast, reliable)
    └─ Research / experimenting?
       └─ Try Lion or Sophia
```

In [ ]:
# Comprehensive comparison table
comparison_data = {
    'Optimizer': [
        'SGD+Momentum',
        'Adam',
        'AdamW',
        'Lion',
        'Sophia',
        '8-bit AdamW',
        '8-bit Lion',
    ],
    'Year': [
        '1999',
        '2014',
        '2017',
        '2023',
        '2023',
        '2021',
        '2023',
    ],
    'Memory/Param': [
        '8 bytes',
        '12 bytes',
        '12 bytes',
        '8 bytes',
        '12 bytes',
        '6 bytes',
        '5 bytes',
    ],
    'Convergence': [
        'Slow',
        'Fast',
        'Fast',
        'Fast',
        'Very Fast',
        'Fast',
        'Fast',
    ],
    'Compute/Step': [
        'Low',
        'Medium',
        'Medium',
        'Medium',
        'High',
        'Medium',
        'Medium',
    ],
    'Best For': [
        'CNNs, simple tasks',
        'General use (legacy)',
        'Transformers, default',
        'LLMs, memory saving',
        'LLM pretraining',
        'Large models, limited GPU',
        'Large models, best memory',
    ],
    'Pros': [
        'Simple, reliable',
        'Fast convergence',
        'Better regularization',
        '33% less memory',
        '2x faster convergence',
        '50% less memory',
        '58% less memory',
    ],
    'Cons': [
        'Slow on transformers',
        'Worse than AdamW',
        'High memory',
        'Needs tuning',
        'More compute',
        'Requires bitsandbytes',
        'Requires bitsandbytes',
    ],
}

df = pd.DataFrame(comparison_data)
print("\nComprehensive Optimizer Comparison:")
print("="*150)
print(df.to_string(index=False))
print("="*150)

print("\n\nRecommendations:")
print("-"*150)
print("🎯 Default Choice: AdamW (lr=1e-4, weight_decay=0.01)")
print("💾 Memory Limited: Lion (lr=1e-5, weight_decay=0.1) or 8-bit AdamW")
print("🚀 Best Performance: Sophia (for long LLM training)")
print("⚡ Best Memory/Speed: 8-bit Lion")
print("🔬 Research: Try Lion or Sophia")

## Summary

**Key Takeaways**:

1. **AdamW** (2017): Standard for transformers
   - Decoupled weight decay
   - 12 bytes per parameter
   - Reliable, well-tested

2. **Lion** (2023): Memory-efficient alternative
   - 33% less memory than Adam
   - Sign-based updates
   - Often better performance
   - Needs 10x smaller LR, 10x larger weight decay

3. **Sophia** (2023): Fastest convergence
   - Uses Hessian information
   - 2x faster convergence on LLMs
   - 50% more compute per step
   - Net speedup for long training

4. **8-bit Optimizers** (2021): Maximum memory savings
   - 75% less memory for optimizer states
   - Works with Adam, Lion, etc.
   - Negligible performance loss
   - Essential for large models

5. **When to Use**:
   - Default: AdamW
   - Memory constrained: Lion or 8-bit optimizers
   - Long LLM training: Sophia
   - Best overall: 8-bit Lion

6. **Memory Comparison** (for 7B model):
   - AdamW: 56 GB
   - Lion: 37 GB (33% less)
   - 8-bit AdamW: 28 GB (50% less)
   - 8-bit Lion: 23 GB (58% less)

**Papers**:
- AdamW: [Decoupled Weight Decay Regularization](https://arxiv.org/abs/1711.05101) (Loshchilov & Hutter, 2017)
- Lion: [Symbolic Discovery of Optimization Algorithms](https://arxiv.org/abs/2302.06675) (Chen et al., 2023)
- Sophia: [Sophia: A Scalable Stochastic Second-order Optimizer](https://arxiv.org/abs/2305.14342) (Liu et al., 2023)
- 8-bit: [8-bit Optimizers via Block-wise Quantization](https://arxiv.org/abs/2110.02861) (Dettmers et al., 2021)